# Amelia / PyTorch：免费 Colab CUDA 分阶段验证模板

**此模板只完成静态结构与代码语法检查，尚未在 Colab 从头到尾运行。** 历史结果来自独立实验，不能当作本 notebook 的运行输出。这里不更改算法、门槛、随机种子或基准计时边界。

固定检出版本：`90699b898a55a9c29f61e01181b8d688ab08b702`，对应迁移前的 `ef729c093e162f4d6ebd797c95a9da8072ac968a`，文件树完全相同。编号映射见 [history](https://github.com/Tocqueville0624/amelia-torch/blob/main/docs/history/README.zh-CN.md)。它**不同于历史云端主计时版本 `905cc79`**；新复现须另存报告，不能替换历史证据。项目为 GPL-3.0-only，三个 UCI 数据集为 CC BY 4.0，来源归属随归档保留。

1. 在 Colab 的 Runtime → Change runtime type 选择 **免费 GPU**，并选择提供 **Python 3.12** 的运行时。实际设备、Python、CPU 配额由后续记录确认；无免费 GPU 就停止，不购买、不切换账户绕过限制。
2. **不要 Run all**。每个大阶段都有默认 `False` 的手动开关；完成、检查并保存该阶段后再启用下一格。保持同一 VM，不在另一 notebook/终端同时进行拟合或压缩备份，不编辑测量源码。
3. 所有结果使用新目录，失败也不覆盖。异常后先保存 checkpoint 与日志，再停止相依步骤。正式 G5 的统计门槛失败可以有完整可审计报告，不能删掉失败或仅选成功集。
4. 不挂载 Google Drive，不索取账户资料；模板不含账号、notebook ID、IP、序列号或运行输出。实际运行日志仍需发布前审查。

流程说明：[cloud-cuda.md](https://github.com/Tocqueville0624/amelia-torch/blob/90699b898a55a9c29f61e01181b8d688ab08b702/docs/cloud-cuda.md)。历史记录见仓库 `docs/validation/2026-09-23-colab-recovered/`、`docs/validation/2026-09-26-colab-native/`；Mac CPU/MPS 证据见 `docs/validation/2026-09-23-development/`，**均不是此 notebook 生成**。


## 1. 新建 checkout，只检查运行环境与源码
这格会下载源码，不安装依赖、不拟合。每次重新实验生成独立目录。保留先前 checkout 和结果；恢复旧实验时先下载，不重新运行此格冒充续跑。


In [ ]:
assert globals().get("ACTIVE_PHASE") is None and globals().get("LIVE_PROCESS_GROUP") is None, "旧任务未确认停止，禁止重置 checkout 或进程标记"

import hashlib
import json
import os
import pathlib
import signal
import subprocess
import sys
import tarfile
import time
import uuid
from datetime import UTC, datetime

assert sys.version_info[:2] == (3, 12), f"需要 Python 3.12，实际为 {sys.version}"
assert sys.platform == "linux", "此模板只用于 Linux Colab；不要在本机执行安装格"
REVISION = "90699b898a55a9c29f61e01181b8d688ab08b702"
RUN_ID = datetime.now(UTC).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid.uuid4().hex[:8]
REPO = pathlib.Path("/content") / ("amelia-torch-" + RUN_ID)
BACKUPS = pathlib.Path("/content") / ("amelia-checkpoints-" + RUN_ID)
assert not REPO.exists() and not BACKUPS.exists()
BACKUPS.mkdir()
with (BACKUPS / "checkout.log").open("x") as log:
    clone = subprocess.run(["git", "clone", "--filter=blob:none",
        "https://github.com/Tocqueville0624/amelia-torch.git", str(REPO)],
        stdout=log, stderr=subprocess.STDOUT, check=False)
    checkout = None
    if clone.returncode == 0:
        checkout = subprocess.run(["git", "checkout", "--detach", REVISION], cwd=REPO,
            stdout=log, stderr=subprocess.STDOUT, check=False)
(BACKUPS / "checkout-status.json").write_text(json.dumps({
    "revision": REVISION, "clone_exit": clone.returncode,
    "checkout_exit": None if checkout is None else checkout.returncode}, indent=2) + "\n")
assert clone.returncode == 0 and checkout is not None and checkout.returncode == 0, "保留 checkout.log 后停止"
os.chdir(REPO)
CLOUD = REPO / "results/local/cloud"
CLOUD.mkdir(parents=True)
PYTHON = str(REPO / ".venv/bin/python")  # 保留 venv 路径，不 resolve 符号链接。
ENV2 = {**os.environ, "R_LIBS_USER": str(REPO / ".R-library"),
        "RETICULATE_PYTHON": PYTHON, "OMP_NUM_THREADS": "2",
        "OPENBLAS_NUM_THREADS": "2", "MKL_NUM_THREADS": "2", "VECLIB_MAXIMUM_THREADS": "2",
        "NVIDIA_TF32_OVERRIDE": "0", "PYTORCH_ENABLE_MPS_FALLBACK": "0", "CUDA_VISIBLE_DEVICES": "0"}
ENV1 = {**ENV2, **{key: "1" for key in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS",
                                      "MKL_NUM_THREADS", "VECLIB_MAXIMUM_THREADS")}}
ACTIVE_PHASE = None
LIVE_PROCESS_GROUP = None
DATASETS = ("covertype", "household_power", "year_prediction_msd")
NATIVE_METHODS = ["r_serial", "r_snow2", "cpu64", "cpu32", "cuda64", "cuda32"]
NATIVE = CLOUD / "native/suite"
HYBRID = CLOUD / "hybrid/suite"
G5_ROUTES = ["r-reference", "hybrid-cpu64", "hybrid-cuda64", "hybrid-cuda32",
             "native-cpu64", "native-cuda64", "native-cuda32"]
PROTOCOL_SHA256 = "7ea7882bccc8d92102a67786bc9c09c2055ce29d97350abfbcf4f74f2187e454"

def process_group_exists(group_id):
    try:
        os.killpg(group_id, 0)
        return True
    except ProcessLookupError:
        return False


def stop_process_group(process):
    # Stop this notebook's command and descendants, then confirm no group remains.
    # Failure to confirm deliberately leaves the notebook blocked, even for zombies.
    for sig in (signal.SIGTERM, signal.SIGKILL):
        try:
            os.killpg(process.pid, sig)
        except ProcessLookupError:
            pass
        deadline = time.monotonic() + 10
        while time.monotonic() < deadline:
            process.poll()  # Reap the group leader when it exits.
            if not process_group_exists(process.pid):
                process.wait()
                return
            time.sleep(0.1)
    raise RuntimeError("Cannot confirm process-group exit; do not checkpoint or start another phase")


def run_logged(command, *, env, log):
    # One thin subprocess helper; numerical behavior remains in existing scripts.
    global LIVE_PROCESS_GROUP
    assert LIVE_PROCESS_GROUP is None, "A previous command group has not been cleared"
    process = subprocess.Popen(command, cwd=REPO, env=env, stdout=log,
                               stderr=subprocess.STDOUT, start_new_session=True)
    LIVE_PROCESS_GROUP = process.pid
    try:
        returncode = process.wait()
    except BaseException:
        stop_process_group(process)
        LIVE_PROCESS_GROUP = None
        raise
    if process_group_exists(process.pid):
        stop_process_group(process)
        LIVE_PROCESS_GROUP = None
        raise RuntimeError("Command returned with live descendants; group was stopped, inspect logs")
    LIVE_PROCESS_GROUP = None
    return subprocess.CompletedProcess(command, returncode)


def require_success(phase):
    assert ACTIVE_PHASE is None and LIVE_PROCESS_GROUP is None, "先确认上一进程组完全退出"
    status = json.loads((CLOUD / phase / "notebook-stage.json").read_text())
    assert status["state"] == "passed", f"先检查 {phase}；失败报告保留，不继续依赖阶段"

def checkpoint(phase):
    # 仅调用已有备份工具；在计时结束后执行，输出放结果目录外，避免递归收集。
    assert ACTIVE_PHASE is None and LIVE_PROCESS_GROUP is None, "未确认进程组退出，禁止 checkpoint"
    label = phase + "-" + datetime.now(UTC).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid.uuid4().hex[:6]
    command = [sys.executable, "scripts/cloud_checkpoint.py", "--phase", phase, "--label", label]
    saved = subprocess.run(command, cwd=REPO, text=True, capture_output=True, check=False)
    (BACKUPS / (label + ".stderr.log")).write_text(saved.stderr)
    (BACKUPS / (label + ".status.json")).write_text(json.dumps({"exit_status": saved.returncode}) + "\n")
    (BACKUPS / (label + ".json")).write_text(saved.stdout)
    print(saved.stdout, flush=True)  # 完整 gzip-base64 envelope，另存 notebook 后可恢复。
    assert saved.returncode == 0, "checkpoint 失败；不要释放 VM，先运行末尾下载格保存原文件"

print({"revision": REVISION, "run_id": RUN_ID, "checkout": str(REPO), "python": sys.version})


## 2. Bootstrap 计划（不安装）
查看输出中的系统包/依赖计划。原版 Amelia 固定 1.8.3 并验证归档 SHA。实际安装单独在下一格。


In [ ]:
RUN_BOOTSTRAP_PLAN = False  # 逐格阅读后仅把本格改为 True；不要 Run all。
assert RUN_BOOTSTRAP_PLAN, "本阶段尚未手动启用"


assert ACTIVE_PHASE is None and LIVE_PROCESS_GROUP is None, "先等待进程组完全退出"
assert subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip() == REVISION
phase = CLOUD / 'bootstrap-plan'
phase.mkdir()  # 已存在即拒绝，不覆盖成功、失败或中断的旧记录。
commands = [[sys.executable, "scripts/cloud_bootstrap.py", "--expected-commit", REVISION]]
records = []
status = {"phase": 'bootstrap-plan', "state": "running", "commands": records}
status_path = phase / "notebook-stage.json"
status_path.write_text(json.dumps(status, indent=2) + "\n")
ACTIVE_PHASE = 'bootstrap-plan'
try:
    for index, command in enumerate(commands):
        record = {"command": [str(arg).replace(str(REPO), "<project>") for arg in command],
                  "exit_status": None, "state": "running", "log": f"{index:02d}.log"}
        records.append(record)
        status_path.write_text(json.dumps(status, indent=2) + "\n")
        print('bootstrap-plan', index + 1, "/", len(commands), record["command"], flush=True)
        with (phase / record["log"]).open("x") as log:
            completed = run_logged(command, env=ENV2, log=log)
        record.update(exit_status=completed.returncode,
                      state="passed" if completed.returncode == 0 else "failed")
        status_path.write_text(json.dumps(status, indent=2) + "\n")
        if completed.returncode != 0:
            raise RuntimeError(f"本阶段退出 {completed.returncode}；完整日志和报告已保留")
    status["state"] = "passed"
except BaseException as error:
    status.update(state="interrupted" if isinstance(error, KeyboardInterrupt) else "failed",
                  error_type=type(error).__name__, uncleared_process_group=LIVE_PROCESS_GROUP)
    if records and records[-1]["state"] == "running":
        records[-1].update(state=status["state"], error_type=type(error).__name__)
    raise
finally:
    status_path.write_text(json.dumps(status, indent=2) + "\n")
    if LIVE_PROCESS_GROUP is not None:
        print("进程组退出尚未确认；保留 ACTIVE_PHASE，不执行备份或后续任务。")
    else:
        ACTIVE_PHASE = None
        checkpoint('bootstrap-plan')  # 即使质量门槛失败，也先备份；失败不触发下一阶段。
print((phase / "00.log").read_text())


## 3. 实际安装（手动启用）
仅此格安装必要系统包与项目依赖；`--install-system-packages` 只用于已授权的这台云端 VM。复用预装 CUDA Torch，不重装 Torch、不采用 Mac lock。共享环境 Ruff 缺 binary 时，已有脚本只在 venv 修复相同版本。失败保留安装日志；不要在原结果目录重试。


In [ ]:
RUN_BOOTSTRAP = False  # 逐格阅读后仅把本格改为 True；不要 Run all。
assert RUN_BOOTSTRAP, "本阶段尚未手动启用"
require_success('bootstrap-plan')

assert ACTIVE_PHASE is None and LIVE_PROCESS_GROUP is None, "先等待进程组完全退出"
assert subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip() == REVISION
phase = CLOUD / 'bootstrap'
phase.mkdir()  # 已存在即拒绝，不覆盖成功、失败或中断的旧记录。
commands = [[sys.executable, "scripts/cloud_bootstrap.py", "--expected-commit", REVISION, "--execute", "--install-system-packages", "--output-dir", str(phase / "installation")]]
records = []
status = {"phase": 'bootstrap', "state": "running", "commands": records}
status_path = phase / "notebook-stage.json"
status_path.write_text(json.dumps(status, indent=2) + "\n")
ACTIVE_PHASE = 'bootstrap'
try:
    for index, command in enumerate(commands):
        record = {"command": [str(arg).replace(str(REPO), "<project>") for arg in command],
                  "exit_status": None, "state": "running", "log": f"{index:02d}.log"}
        records.append(record)
        status_path.write_text(json.dumps(status, indent=2) + "\n")
        print('bootstrap', index + 1, "/", len(commands), record["command"], flush=True)
        with (phase / record["log"]).open("x") as log:
            completed = run_logged(command, env=ENV2, log=log)
        record.update(exit_status=completed.returncode,
                      state="passed" if completed.returncode == 0 else "failed")
        status_path.write_text(json.dumps(status, indent=2) + "\n")
        if completed.returncode != 0:
            raise RuntimeError(f"本阶段退出 {completed.returncode}；完整日志和报告已保留")
    status["state"] = "passed"
except BaseException as error:
    status.update(state="interrupted" if isinstance(error, KeyboardInterrupt) else "failed",
                  error_type=type(error).__name__, uncleared_process_group=LIVE_PROCESS_GROUP)
    if records and records[-1]["state"] == "running":
        records[-1].update(state=status["state"], error_type=type(error).__name__)
    raise
finally:
    status_path.write_text(json.dumps(status, indent=2) + "\n")
    if LIVE_PROCESS_GROUP is not None:
        print("进程组退出尚未确认；保留 ACTIVE_PHASE，不执行备份或后续任务。")
    else:
        ACTIVE_PHASE = None
        checkpoint('bootstrap')  # 即使质量门槛失败，也先备份；失败不触发下一阶段。


## 4. CUDA 探针、Python 与 9 个 R 测试文件、下游示例和固定小型对照
此格会进行正确性拟合，但不作性能测量。全部子进程保存 stdout/stderr 与退出状态，第一次失败后停止本格其余检查。九个 R 文件来自固定源码，不用历史五/七文件结果替代。CUDA float64/32 都显式测试；新增公开边界检查保留既有阈值。


In [ ]:
RUN_VALIDATION = False  # 逐格阅读后仅把本格改为 True；不要 Run all。
assert RUN_VALIDATION, "本阶段尚未手动启用"
require_success('bootstrap')

assert ACTIVE_PHASE is None and LIVE_PROCESS_GROUP is None, "先等待进程组完全退出"
assert subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip() == REVISION
phase = CLOUD / 'validation'
phase.mkdir()  # 已存在即拒绝，不覆盖成功、失败或中断的旧记录。
commands = [
    [PYTHON, "-m", "amelia_torch.diagnostics", "--require-device", "cuda", "--output", str(phase / "cuda-probe.json")],
    [PYTHON, "-m", "ruff", "--version"],
    [PYTHON, "-m", "ruff", "check", "src", "tests", "scripts", "examples"],
    [PYTHON, "-m", "pytest", "-q"],
    *[["Rscript", "r-package/tests/" + case + ".R"] for case in (
        "bridge", "compatibility", "torch-compat", "public-edge-cases", "reference_metadata",
        "downstream", "downstream_extended", "autopri-boundary", "reference-parallel")],
    [PYTHON, "examples/python_r_downstream.py", "--output", str(phase / "downstream-example")],
    *[command for dtype in ("float64", "float32") for command in (
        [PYTHON, "scripts/validate_accelerator.py", "--device", "cuda", "--dtype", dtype,
         "--output", str(phase / ("native-" + dtype + ".json"))],
        ["Rscript", "scripts/validate_r_accelerator.R", "cuda", dtype,
         str(phase / ("hybrid-" + dtype + ".json"))],
        ["Rscript", "scripts/validate_r_accelerator_edges.R", "cuda", dtype,
         str(phase / ("hybrid-edges-" + dtype + ".json"))])]
]
records = []
status = {"phase": 'validation', "state": "running", "commands": records}
status_path = phase / "notebook-stage.json"
status_path.write_text(json.dumps(status, indent=2) + "\n")
ACTIVE_PHASE = 'validation'
try:
    for index, command in enumerate(commands):
        record = {"command": [str(arg).replace(str(REPO), "<project>") for arg in command],
                  "exit_status": None, "state": "running", "log": f"{index:02d}.log"}
        records.append(record)
        status_path.write_text(json.dumps(status, indent=2) + "\n")
        print('validation', index + 1, "/", len(commands), record["command"], flush=True)
        with (phase / record["log"]).open("x") as log:
            completed = run_logged(command, env=ENV1, log=log)
        record.update(exit_status=completed.returncode,
                      state="passed" if completed.returncode == 0 else "failed")
        status_path.write_text(json.dumps(status, indent=2) + "\n")
        if completed.returncode != 0:
            raise RuntimeError(f"本阶段退出 {completed.returncode}；完整日志和报告已保留")
    status["state"] = "passed"
except BaseException as error:
    status.update(state="interrupted" if isinstance(error, KeyboardInterrupt) else "failed",
                  error_type=type(error).__name__, uncleared_process_group=LIVE_PROCESS_GROUP)
    if records and records[-1]["state"] == "running":
        records[-1].update(state=status["state"], error_type=type(error).__name__)
    raise
finally:
    status_path.write_text(json.dumps(status, indent=2) + "\n")
    if LIVE_PROCESS_GROUP is not None:
        print("进程组退出尚未确认；保留 ACTIVE_PHASE，不执行备份或后续任务。")
    else:
        ACTIVE_PHASE = None
        checkpoint('validation')  # 即使质量门槛失败，也先备份；失败不触发下一阶段。


## 5. 三组数据：下载校验与固定人工缺失输入
下载器核对官方压缩包大小、SHA-256、ZIP CRC 并预检磁盘空间。三个压缩包约 232 MiB；保留至少 5 GiB。每组从完整可观察行固定抽取 100,000 行，再加 8 模式块状 MCAR，目标 30%（Household 实际 2/7）。不把人工 heldout 与天然缺失混淆；90 维 YearPrediction 不含年份目标。这里不是全量吞吐或逐格 MCAR 压力测试。


In [ ]:
RUN_DATA = False  # 逐格阅读后仅把本格改为 True；不要 Run all。
assert RUN_DATA, "本阶段尚未手动启用"
require_success('validation')

assert ACTIVE_PHASE is None and LIVE_PROCESS_GROUP is None, "先等待进程组完全退出"
assert subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip() == REVISION
phase = CLOUD / 'data'
phase.mkdir()  # 已存在即拒绝，不覆盖成功、失败或中断的旧记录。
commands = [
    [PYTHON, "scripts/download_datasets.py", "--datasets", "all", "--max-download-mib", "300"],
    *[[PYTHON, "scripts/prepare_benchmark_data.py", "--dataset", dataset, "--rows", "100000",
       "--sampling", "random", "--mechanism", "block_mcar", "--patterns", "8",
       "--missing-rate", "0.3", "--seed", "20260923", "--output",
       "data/prepared/" + dataset + "-n100000-block_mcar-rate30-seed20260923.npz"] for dataset in DATASETS]
]
records = []
status = {"phase": 'data', "state": "running", "commands": records}
status_path = phase / "notebook-stage.json"
status_path.write_text(json.dumps(status, indent=2) + "\n")
ACTIVE_PHASE = 'data'
try:
    for index, command in enumerate(commands):
        record = {"command": [str(arg).replace(str(REPO), "<project>") for arg in command],
                  "exit_status": None, "state": "running", "log": f"{index:02d}.log"}
        records.append(record)
        status_path.write_text(json.dumps(status, indent=2) + "\n")
        print('data', index + 1, "/", len(commands), record["command"], flush=True)
        with (phase / record["log"]).open("x") as log:
            completed = run_logged(command, env=ENV2, log=log)
        record.update(exit_status=completed.returncode,
                      state="passed" if completed.returncode == 0 else "failed")
        status_path.write_text(json.dumps(status, indent=2) + "\n")
        if completed.returncode != 0:
            raise RuntimeError(f"本阶段退出 {completed.returncode}；完整日志和报告已保留")
    status["state"] = "passed"
except BaseException as error:
    status.update(state="interrupted" if isinstance(error, KeyboardInterrupt) else "failed",
                  error_type=type(error).__name__, uncleared_process_group=LIVE_PROCESS_GROUP)
    if records and records[-1]["state"] == "running":
        records[-1].update(state=status["state"], error_type=type(error).__name__)
    raise
finally:
    status_path.write_text(json.dumps(status, indent=2) + "\n")
    if LIVE_PROCESS_GROUP is not None:
        print("进程组退出尚未确认；保留 ACTIVE_PHASE，不执行备份或后续任务。")
    else:
        ACTIVE_PHASE = None
        checkpoint('data')  # 即使质量门槛失败，也先备份；失败不触发下一阶段。


## 6. 双核预算断言：尚不计时
该计划专用于有效 CPU 配额恰为 2 的 VM；读取逻辑 CPU、affinity 与 cgroup 限额。若不同或无法判断，不自动改预算、不继续。应另行预定同机所有方法的统一预算并记录。原 Mac 四线程报告不改变。


In [ ]:
require_success("data")
assert ACTIVE_PHASE is None and LIVE_PROCESS_GROUP is None
phase = CLOUD / "budget"
phase.mkdir()
counts = {"logical": os.cpu_count(), "affinity": len(os.sched_getaffinity(0))}
cgroup = pathlib.Path("/sys/fs/cgroup/cpu.max")
quota_text = cgroup.read_text().strip() if cgroup.exists() else None
quota = None
if quota_text:
    quota_raw, period_raw = quota_text.split()
    if quota_raw != "max":
        quota = int(quota_raw) / int(period_raw)
else:
    qpath = pathlib.Path("/sys/fs/cgroup/cpu/cpu.cfs_quota_us")
    ppath = pathlib.Path("/sys/fs/cgroup/cpu/cpu.cfs_period_us")
    if qpath.exists() and ppath.exists() and int(qpath.read_text()) > 0:
        quota = int(qpath.read_text()) / int(ppath.read_text())
effective = min([value for value in counts.values() if value is not None] +
                ([] if quota is None else [quota]))
budget = {"visible_cpu_counts": counts, "cgroup_v2_cpu_max": quota_text,
          "cgroup_quota_cores": quota, "effective_cores": effective, "threads": 2, "snow_workers": 2,
          "native_configurations": 3 * len(NATIVE_METHODS), "hybrid_configurations": 3 * 4,
          "warmups": 2, "measured_repeats": 5, "imputations_per_fit": 5,
          "same_vm_required": True, "no_concurrent_fit_or_checkpoint": True}
(phase / "budget.json").write_text(json.dumps(budget, indent=2) + "\n")
status = {"phase": "budget", "state": "passed" if effective == 2 else "failed"}
(phase / "notebook-stage.json").write_text(json.dumps(status) + "\n")
checkpoint("budget")
assert effective == 2, "该模板固定双核预算；实际配额不同，停止并另行制定完整公平计划"
print(budget)


## 7. Native 主性能套件：18 个配置（手动长任务）
三数据 × R serial / R snow2 / CPU64 / CPU32 / CUDA64 / CUDA32。每配置 2 次预热 + 5 次正式重复，每次 m=5；需要较长免费会话，不能保证额度足够。原版 R 与 NumPy PCG64 的同整数 seed 不代表同抽样。计时同步、数据传输和输出边界由现有 runner 固定。


In [ ]:
RUN_NATIVE = False  # 逐格阅读后仅把本格改为 True；不要 Run all。
assert RUN_NATIVE, "本阶段尚未手动启用"
require_success('budget')

assert ACTIVE_PHASE is None and LIVE_PROCESS_GROUP is None, "先等待进程组完全退出"
assert subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip() == REVISION
phase = CLOUD / 'native'
phase.mkdir()  # 已存在即拒绝，不覆盖成功、失败或中断的旧记录。
commands = [[PYTHON, "scripts/run_benchmark_suite.py", "--gpu", "cuda", "--output-dir", str(NATIVE), "--rows", "100000", "--m", "5", "--warmups", "2", "--repeats", "5", "--threads", "2", "--workers", "2"]]
records = []
status = {"phase": 'native', "state": "running", "commands": records}
status_path = phase / "notebook-stage.json"
status_path.write_text(json.dumps(status, indent=2) + "\n")
ACTIVE_PHASE = 'native'
try:
    for index, command in enumerate(commands):
        record = {"command": [str(arg).replace(str(REPO), "<project>") for arg in command],
                  "exit_status": None, "state": "running", "log": f"{index:02d}.log"}
        records.append(record)
        status_path.write_text(json.dumps(status, indent=2) + "\n")
        print('native', index + 1, "/", len(commands), record["command"], flush=True)
        with (phase / record["log"]).open("x") as log:
            completed = run_logged(command, env=ENV2, log=log)
        record.update(exit_status=completed.returncode,
                      state="passed" if completed.returncode == 0 else "failed")
        status_path.write_text(json.dumps(status, indent=2) + "\n")
        if completed.returncode != 0:
            raise RuntimeError(f"本阶段退出 {completed.returncode}；完整日志和报告已保留")
    status["state"] = "passed"
except BaseException as error:
    status.update(state="interrupted" if isinstance(error, KeyboardInterrupt) else "failed",
                  error_type=type(error).__name__, uncleared_process_group=LIVE_PROCESS_GROUP)
    if records and records[-1]["state"] == "running":
        records[-1].update(state=status["state"], error_type=type(error).__name__)
    raise
finally:
    status_path.write_text(json.dumps(status, indent=2) + "\n")
    if LIVE_PROCESS_GROUP is not None:
        print("进程组退出尚未确认；保留 ACTIVE_PHASE，不执行备份或后续任务。")
    else:
        ACTIVE_PHASE = None
        checkpoint('native')  # 即使质量门槛失败，也先备份；失败不触发下一阶段。


## 8. Native 离线审计与 12 份参数文件备份（无拟合）
主任务中断也可以单独运行审计以记录不完整；不允许只按退出码或只挑成功 repeat 宣称加速。JSON checkpoint **不包含 NPZ**。审计后把全部 12 个参数文件的原字节另行 base64 备份，保存原 SHA；这些是首次正式 repeat 的 theta 和首 1000 行样本，不是完整矩阵。备份在计时外执行，不能算入性能时间。


In [ ]:
RUN_NATIVE_AUDIT = False  # 逐格阅读后仅把本格改为 True；不要 Run all。
assert RUN_NATIVE_AUDIT, "本阶段尚未手动启用"

assert (NATIVE / "suite.json").exists(), "尚无主报告可审计"
assert ACTIVE_PHASE is None and LIVE_PROCESS_GROUP is None, "先等待进程组完全退出"
assert subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip() == REVISION
phase = CLOUD / 'native-audit'
phase.mkdir()  # 已存在即拒绝，不覆盖成功、失败或中断的旧记录。
commands = [[PYTHON, "scripts/summarize_benchmarks.py", "--input-dir", str(NATIVE), "--output-dir", str(phase / "export"), "--methods", *NATIVE_METHODS]]
records = []
status = {"phase": 'native-audit', "state": "running", "commands": records}
status_path = phase / "notebook-stage.json"
status_path.write_text(json.dumps(status, indent=2) + "\n")
ACTIVE_PHASE = 'native-audit'
try:
    for index, command in enumerate(commands):
        record = {"command": [str(arg).replace(str(REPO), "<project>") for arg in command],
                  "exit_status": None, "state": "running", "log": f"{index:02d}.log"}
        records.append(record)
        status_path.write_text(json.dumps(status, indent=2) + "\n")
        print('native-audit', index + 1, "/", len(commands), record["command"], flush=True)
        with (phase / record["log"]).open("x") as log:
            completed = run_logged(command, env=ENV2, log=log)
        record.update(exit_status=completed.returncode,
                      state="passed" if completed.returncode == 0 else "failed")
        status_path.write_text(json.dumps(status, indent=2) + "\n")
        if completed.returncode != 0:
            raise RuntimeError(f"本阶段退出 {completed.returncode}；完整日志和报告已保留")
    status["state"] = "passed"
except BaseException as error:
    status.update(state="interrupted" if isinstance(error, KeyboardInterrupt) else "failed",
                  error_type=type(error).__name__, uncleared_process_group=LIVE_PROCESS_GROUP)
    if records and records[-1]["state"] == "running":
        records[-1].update(state=status["state"], error_type=type(error).__name__)
    raise
finally:
    status_path.write_text(json.dumps(status, indent=2) + "\n")
    if LIVE_PROCESS_GROUP is not None:
        print("进程组退出尚未确认；保留 ACTIVE_PHASE，不执行备份或后续任务。")
    else:
        ACTIVE_PHASE = None
        checkpoint('native-audit')  # 即使质量门槛失败，也先备份；失败不触发下一阶段。


In [ ]:
RUN_PARAMETER_BACKUP = False
assert RUN_PARAMETER_BACKUP, "手动启用；只读已有 NPZ，不拟合"
require_success("native-audit")
assert ACTIVE_PHASE is None and LIVE_PROCESS_GROUP is None
import base64

expected = {f"{dataset}-{method}.parameters.npz" for dataset in DATASETS
            for method in ("cpu64", "cpu32", "cuda64", "cuda32")}
paths = sorted(NATIVE.glob("*.parameters.npz"))
assert {path.name for path in paths} == expected and len(paths) == 12
hashes = {}
for path in paths:
    raw = path.read_bytes()
    digest = hashlib.sha256(raw).hexdigest()
    hashes[path.name] = digest
    envelope = {"kind": "native-parameter-raw-base64-v1", "filename": path.name,
                "bytes": len(raw), "sha256": digest, "data": base64.b64encode(raw).decode()}
    destination = BACKUPS / (path.name + ".json")
    with destination.open("x") as stream:
        stream.write(json.dumps(envelope) + "\n")
    print(json.dumps(envelope), flush=True)
with (BACKUPS / "native-parameter-hashes.json").open("x") as stream:
    json.dump(hashes, stream, indent=2)
print("已输出 12 个原始二进制 envelope；保存 notebook，并另用末尾下载格下载归档。")


## 9. R hybrid 完整流程：12 个配置（手动长任务）
仅 native 审计通过后执行。三数据 × CPU64 / CPU32 / CUDA64 / CUDA32，复用同一 CSV、m=5、2 预热 + 5 正式、2 线程。原 R 预处理、bootstrap、draw 与后处理保留，EM 使用 Torch，R 桥接及转换计时在内。外部进程冷启动、安装、读取和评分在外；不能当作每种真实调用方式的总延迟。


In [ ]:
RUN_HYBRID = False  # 逐格阅读后仅把本格改为 True；不要 Run all。
assert RUN_HYBRID, "本阶段尚未手动启用"
require_success('native-audit')
require_success('budget')

assert ACTIVE_PHASE is None and LIVE_PROCESS_GROUP is None, "先等待进程组完全退出"
assert subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip() == REVISION
phase = CLOUD / 'hybrid'
phase.mkdir()  # 已存在即拒绝，不覆盖成功、失败或中断的旧记录。
commands = [[PYTHON, "scripts/run_hybrid_suite.py", "--methods", "cpu64", "cpu32", "cuda64", "cuda32", "--csv-dir", str(NATIVE / "inputs"), "--output-dir", str(HYBRID), "--rows", "100000", "--m", "5", "--warmups", "2", "--repeats", "5", "--threads", "2"]]
records = []
status = {"phase": 'hybrid', "state": "running", "commands": records}
status_path = phase / "notebook-stage.json"
status_path.write_text(json.dumps(status, indent=2) + "\n")
ACTIVE_PHASE = 'hybrid'
try:
    for index, command in enumerate(commands):
        record = {"command": [str(arg).replace(str(REPO), "<project>") for arg in command],
                  "exit_status": None, "state": "running", "log": f"{index:02d}.log"}
        records.append(record)
        status_path.write_text(json.dumps(status, indent=2) + "\n")
        print('hybrid', index + 1, "/", len(commands), record["command"], flush=True)
        with (phase / record["log"]).open("x") as log:
            completed = run_logged(command, env=ENV2, log=log)
        record.update(exit_status=completed.returncode,
                      state="passed" if completed.returncode == 0 else "failed")
        status_path.write_text(json.dumps(status, indent=2) + "\n")
        if completed.returncode != 0:
            raise RuntimeError(f"本阶段退出 {completed.returncode}；完整日志和报告已保留")
    status["state"] = "passed"
except BaseException as error:
    status.update(state="interrupted" if isinstance(error, KeyboardInterrupt) else "failed",
                  error_type=type(error).__name__, uncleared_process_group=LIVE_PROCESS_GROUP)
    if records and records[-1]["state"] == "running":
        records[-1].update(state=status["state"], error_type=type(error).__name__)
    raise
finally:
    status_path.write_text(json.dumps(status, indent=2) + "\n")
    if LIVE_PROCESS_GROUP is not None:
        print("进程组退出尚未确认；保留 ACTIVE_PHASE，不执行备份或后续任务。")
    else:
        ACTIVE_PHASE = None
        checkpoint('hybrid')  # 即使质量门槛失败，也先备份；失败不触发下一阶段。


## 10. Hybrid 离线审计、两类有限配对（无拟合）
Hybrid 对原版 R 使用同 R RNG/seed 的已保存均值、协方差、迭代、RMSE 摘要；摘要接近不能证明逐值一致。Native CPU/GPU 对照只比较同 dtype、首次正式 repeat、首 1000 行与 theta，保留原单位及 truth 列 SD 标准化差。保存的 bootstrap attempts/seed 不等于已经保存完整随机输入；不设新的等价阈值。


In [ ]:
RUN_HYBRID_AUDIT = False  # 逐格阅读后仅把本格改为 True；不要 Run all。
assert RUN_HYBRID_AUDIT, "本阶段尚未手动启用"

assert (HYBRID / "suite.json").exists(), "尚无 hybrid 主报告可审计"
assert ACTIVE_PHASE is None and LIVE_PROCESS_GROUP is None, "先等待进程组完全退出"
assert subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip() == REVISION
phase = CLOUD / 'hybrid-audit'
phase.mkdir()  # 已存在即拒绝，不覆盖成功、失败或中断的旧记录。
commands = [[PYTHON, "scripts/summarize_hybrid.py", "--input-dir", str(HYBRID), "--output-dir", str(phase / "export")]]
records = []
status = {"phase": 'hybrid-audit', "state": "running", "commands": records}
status_path = phase / "notebook-stage.json"
status_path.write_text(json.dumps(status, indent=2) + "\n")
ACTIVE_PHASE = 'hybrid-audit'
try:
    for index, command in enumerate(commands):
        record = {"command": [str(arg).replace(str(REPO), "<project>") for arg in command],
                  "exit_status": None, "state": "running", "log": f"{index:02d}.log"}
        records.append(record)
        status_path.write_text(json.dumps(status, indent=2) + "\n")
        print('hybrid-audit', index + 1, "/", len(commands), record["command"], flush=True)
        with (phase / record["log"]).open("x") as log:
            completed = run_logged(command, env=ENV2, log=log)
        record.update(exit_status=completed.returncode,
                      state="passed" if completed.returncode == 0 else "failed")
        status_path.write_text(json.dumps(status, indent=2) + "\n")
        if completed.returncode != 0:
            raise RuntimeError(f"本阶段退出 {completed.returncode}；完整日志和报告已保留")
    status["state"] = "passed"
except BaseException as error:
    status.update(state="interrupted" if isinstance(error, KeyboardInterrupt) else "failed",
                  error_type=type(error).__name__, uncleared_process_group=LIVE_PROCESS_GROUP)
    if records and records[-1]["state"] == "running":
        records[-1].update(state=status["state"], error_type=type(error).__name__)
    raise
finally:
    status_path.write_text(json.dumps(status, indent=2) + "\n")
    if LIVE_PROCESS_GROUP is not None:
        print("进程组退出尚未确认；保留 ACTIVE_PHASE，不执行备份或后续任务。")
    else:
        ACTIVE_PHASE = None
        checkpoint('hybrid-audit')  # 即使质量门槛失败，也先备份；失败不触发下一阶段。


In [ ]:
RUN_PAIRED_COMPARISONS = False  # 逐格阅读后仅把本格改为 True；不要 Run all。
assert RUN_PAIRED_COMPARISONS, "本阶段尚未手动启用"
require_success('native-audit')
require_success('hybrid-audit')
assert (BACKUPS / "native-parameter-hashes.json").exists(), "先保存原 NPZ 的 12 项哈希"
assert ACTIVE_PHASE is None and LIVE_PROCESS_GROUP is None, "先等待进程组完全退出"
assert subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip() == REVISION
phase = CLOUD / 'paired-comparisons'
phase.mkdir()  # 已存在即拒绝，不覆盖成功、失败或中断的旧记录。
commands = [
    [PYTHON, "scripts/compare_hybrid_reference.py", "--reference-dir", str(NATIVE),
     "--hybrid-dir", str(HYBRID), "--reference-methods", *NATIVE_METHODS,
     "--output", str(phase / "hybrid-reference.json")],
    [PYTHON, "scripts/compare_native_parameters.py", "--input-dir", str(NATIVE),
     "--prepared-dir", "data/prepared", "--gpu", "cuda", "--r-workers", "2",
     "--parameter-hashes", str(BACKUPS / "native-parameter-hashes.json"),
     "--output", str(phase / "native-parameters.json")]
]
records = []
status = {"phase": 'paired-comparisons', "state": "running", "commands": records}
status_path = phase / "notebook-stage.json"
status_path.write_text(json.dumps(status, indent=2) + "\n")
ACTIVE_PHASE = 'paired-comparisons'
try:
    for index, command in enumerate(commands):
        record = {"command": [str(arg).replace(str(REPO), "<project>") for arg in command],
                  "exit_status": None, "state": "running", "log": f"{index:02d}.log"}
        records.append(record)
        status_path.write_text(json.dumps(status, indent=2) + "\n")
        print('paired-comparisons', index + 1, "/", len(commands), record["command"], flush=True)
        with (phase / record["log"]).open("x") as log:
            completed = run_logged(command, env=ENV2, log=log)
        record.update(exit_status=completed.returncode,
                      state="passed" if completed.returncode == 0 else "failed")
        status_path.write_text(json.dumps(status, indent=2) + "\n")
        if completed.returncode != 0:
            raise RuntimeError(f"本阶段退出 {completed.returncode}；完整日志和报告已保留")
    status["state"] = "passed"
except BaseException as error:
    status.update(state="interrupted" if isinstance(error, KeyboardInterrupt) else "failed",
                  error_type=type(error).__name__, uncleared_process_group=LIVE_PROCESS_GROUP)
    if records and records[-1]["state"] == "running":
        records[-1].update(state=status["state"], error_type=type(error).__name__)
    raise
finally:
    status_path.write_text(json.dumps(status, indent=2) + "\n")
    if LIVE_PROCESS_GROUP is not None:
        print("进程组退出尚未确认；保留 ACTIVE_PHASE，不执行备份或后续任务。")
    else:
        ACTIVE_PHASE = None
        checkpoint('paired-comparisons')  # 即使质量门槛失败，也先备份；失败不触发下一阶段。


## 11. 冻结 G5：七路线 smoke（手动小型流程检查）
先核对原预注册文件 SHA。七路线为原版 R、hybrid CPU64/CUDA64/CUDA32、native CPU64/CUDA64/CUDA32。smoke 每路线 3 MCAR + 3 MAR + 2 stress，只检查流程，不足以通过正式推断门槛。每库 1 线程，不与性能任务并发，不作速度结论。

smoke 总预算固定为 600 秒；不以超时后的成功子集声称流程通过。


In [ ]:
RUN_G5_SMOKE = False  # 逐格阅读后仅把本格改为 True；不要 Run all。
assert RUN_G5_SMOKE, "本阶段尚未手动启用"
require_success('validation')
assert hashlib.sha256((REPO / "docs/validation/g5-prespecified/protocol.json").read_bytes()).hexdigest() == PROTOCOL_SHA256
assert ACTIVE_PHASE is None and LIVE_PROCESS_GROUP is None, "先等待进程组完全退出"
assert subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip() == REVISION
phase = CLOUD / 'g5-smoke'
phase.mkdir()  # 已存在即拒绝，不覆盖成功、失败或中断的旧记录。
commands = [[PYTHON, "scripts/validate_inference_suite.py", "--mode", "smoke", "--routes", *G5_ROUTES, "--output", str(phase / "run"), "--time-budget-seconds", "600"]]
records = []
status = {"phase": 'g5-smoke', "state": "running", "commands": records}
status_path = phase / "notebook-stage.json"
status_path.write_text(json.dumps(status, indent=2) + "\n")
ACTIVE_PHASE = 'g5-smoke'
try:
    for index, command in enumerate(commands):
        record = {"command": [str(arg).replace(str(REPO), "<project>") for arg in command],
                  "exit_status": None, "state": "running", "log": f"{index:02d}.log"}
        records.append(record)
        status_path.write_text(json.dumps(status, indent=2) + "\n")
        print('g5-smoke', index + 1, "/", len(commands), record["command"], flush=True)
        with (phase / record["log"]).open("x") as log:
            completed = run_logged(command, env=ENV1, log=log)
        record.update(exit_status=completed.returncode,
                      state="passed" if completed.returncode == 0 else "failed")
        status_path.write_text(json.dumps(status, indent=2) + "\n")
        if completed.returncode != 0:
            raise RuntimeError(f"本阶段退出 {completed.returncode}；完整日志和报告已保留")
    status["state"] = "passed"
except BaseException as error:
    status.update(state="interrupted" if isinstance(error, KeyboardInterrupt) else "failed",
                  error_type=type(error).__name__, uncleared_process_group=LIVE_PROCESS_GROUP)
    if records and records[-1]["state"] == "running":
        records[-1].update(state=status["state"], error_type=type(error).__name__)
    raise
finally:
    status_path.write_text(json.dumps(status, indent=2) + "\n")
    if LIVE_PROCESS_GROUP is not None:
        print("进程组退出尚未确认；保留 ACTIVE_PHASE，不执行备份或后续任务。")
    else:
        ACTIVE_PHASE = None
        checkpoint('g5-smoke')  # 即使质量门槛失败，也先备份；失败不触发下一阶段。


## 12. 原冻结 G5 正式实验（手动长任务，7200 秒总预算）
每路线 200 MCAR + 200 基于始终观察 y 的 MAR + 20 独立 stress，n=300、m=5；七路线共 2,940 个模拟数据集拟合。保持原模型/种子/R seed 映射/门槛与 Rubin 公式。stress 只做描述，不套 coverage 门槛。**不包含未批准的 1000 MCAR 后续方案**。

正式进程可能因统计区间未落入预定边界退出 1，即使每个拟合均成功。先保存全部报告，再运行下一格独立审计；不能改 seed、扩大容忍或只报成功集。原版 R 自身未过 absolute gate 也须保留。7200 秒是资源上限，不是性能结果；超时/中断不能视为完整验收。


In [ ]:
RUN_G5_FORMAL = False  # 逐格阅读后仅把本格改为 True；不要 Run all。
assert RUN_G5_FORMAL, "本阶段尚未手动启用"
require_success('g5-smoke')
assert hashlib.sha256((REPO / "docs/validation/g5-prespecified/protocol.json").read_bytes()).hexdigest() == PROTOCOL_SHA256
assert ACTIVE_PHASE is None and LIVE_PROCESS_GROUP is None, "先等待进程组完全退出"
assert subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip() == REVISION
phase = CLOUD / 'g5-formal'
phase.mkdir()  # 已存在即拒绝，不覆盖成功、失败或中断的旧记录。
commands = [[PYTHON, "scripts/validate_inference_suite.py", "--mode", "formal", "--routes", *G5_ROUTES, "--output", str(phase / "run"), "--time-budget-seconds", "7200"]]
records = []
status = {"phase": 'g5-formal', "state": "running", "commands": records}
status_path = phase / "notebook-stage.json"
status_path.write_text(json.dumps(status, indent=2) + "\n")
ACTIVE_PHASE = 'g5-formal'
try:
    for index, command in enumerate(commands):
        record = {"command": [str(arg).replace(str(REPO), "<project>") for arg in command],
                  "exit_status": None, "state": "running", "log": f"{index:02d}.log"}
        records.append(record)
        status_path.write_text(json.dumps(status, indent=2) + "\n")
        print('g5-formal', index + 1, "/", len(commands), record["command"], flush=True)
        with (phase / record["log"]).open("x") as log:
            completed = run_logged(command, env=ENV1, log=log)
        record.update(exit_status=completed.returncode,
                      state="passed" if completed.returncode == 0 else "failed")
        status_path.write_text(json.dumps(status, indent=2) + "\n")
        if completed.returncode != 0:
            raise RuntimeError(f"本阶段退出 {completed.returncode}；完整日志和报告已保留")
    status["state"] = "passed"
except BaseException as error:
    status.update(state="interrupted" if isinstance(error, KeyboardInterrupt) else "failed",
                  error_type=type(error).__name__, uncleared_process_group=LIVE_PROCESS_GROUP)
    if records and records[-1]["state"] == "running":
        records[-1].update(state=status["state"], error_type=type(error).__name__)
    raise
finally:
    status_path.write_text(json.dumps(status, indent=2) + "\n")
    if LIVE_PROCESS_GROUP is not None:
        print("进程组退出尚未确认；保留 ACTIVE_PHASE，不执行备份或后续任务。")
    else:
        ACTIVE_PHASE = None
        checkpoint('g5-formal')  # 即使质量门槛失败，也先备份；失败不触发下一阶段。


## 13. G5 独立审计：统计失败也照常读取（无拟合）
此格不要求 formal 退出 0，只要求已有报告。在当前 VM、相同环境下审计，避免在另一平台重新生成输入而产生浮点哈希差别。`audit_passed` 表示报告/输入/门槛计算经核查；`all_primary_criteria_passed` 才是原统计标准的结论，两者不可混写。失败或证据不足都保留。


In [ ]:
RUN_G5_AUDIT = False  # 逐格阅读后仅把本格改为 True；不要 Run all。
assert RUN_G5_AUDIT, "本阶段尚未手动启用"

assert (CLOUD / "g5-formal/run/inference-suite.json").exists(), "尚无正式报告；先保存中断日志"
assert ACTIVE_PHASE is None and LIVE_PROCESS_GROUP is None, "先等待进程组完全退出"
assert subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip() == REVISION
phase = CLOUD / 'g5-audit'
phase.mkdir()  # 已存在即拒绝，不覆盖成功、失败或中断的旧记录。
commands = [[PYTHON, "scripts/summarize_inference_suite.py", str(CLOUD / "g5-formal/run/inference-suite.json"), "--output", str(phase / "export")]]
records = []
status = {"phase": 'g5-audit', "state": "running", "commands": records}
status_path = phase / "notebook-stage.json"
status_path.write_text(json.dumps(status, indent=2) + "\n")
ACTIVE_PHASE = 'g5-audit'
try:
    for index, command in enumerate(commands):
        record = {"command": [str(arg).replace(str(REPO), "<project>") for arg in command],
                  "exit_status": None, "state": "running", "log": f"{index:02d}.log"}
        records.append(record)
        status_path.write_text(json.dumps(status, indent=2) + "\n")
        print('g5-audit', index + 1, "/", len(commands), record["command"], flush=True)
        with (phase / record["log"]).open("x") as log:
            completed = run_logged(command, env=ENV1, log=log)
        record.update(exit_status=completed.returncode,
                      state="passed" if completed.returncode == 0 else "failed")
        status_path.write_text(json.dumps(status, indent=2) + "\n")
        if completed.returncode != 0:
            raise RuntimeError(f"本阶段退出 {completed.returncode}；完整日志和报告已保留")
    status["state"] = "passed"
except BaseException as error:
    status.update(state="interrupted" if isinstance(error, KeyboardInterrupt) else "failed",
                  error_type=type(error).__name__, uncleared_process_group=LIVE_PROCESS_GROUP)
    if records and records[-1]["state"] == "running":
        records[-1].update(state=status["state"], error_type=type(error).__name__)
    raise
finally:
    status_path.write_text(json.dumps(status, indent=2) + "\n")
    if LIVE_PROCESS_GROUP is not None:
        print("进程组退出尚未确认；保留 ACTIVE_PHASE，不执行备份或后续任务。")
    else:
        ACTIVE_PHASE = None
        checkpoint('g5-audit')  # 即使质量门槛失败，也先备份；失败不触发下一阶段。


## 14. 随时重做阶段 checkpoint（不重跑计算）
只在所有拟合停止后执行。按已存在阶段分开备份，避免单包超过 64 MiB 解压上限。完整 envelope 输出与外部备份文件都保存；备份工具不是实时外部存储，VM 在阶段中丢失仍可能失去最后一个 checkpoint 后的记录。它不保证恢复未写盘结果、SIGKILL 终态、NPZ 或 RDS。

每条实验命令在独立进程组中启动。可捕获中断会先终止该组及其继承进程并确认退出，之后才备份。若无法确认退出（包括遗留僵尸进程），模板保留阻塞状态，不开始备份或新的拟合；先检查并清理该实验进程组，无法确认就保持停止。该检查只覆盖模板启动且未主动脱离进程组的命令，不能证明另一 notebook/终端没有后台任务。整台 VM 消失仍无法执行清理。


In [ ]:
PHASE_TO_SAVE = "validation"  # 改为要保存的已有直接阶段名，例如 native / hybrid / g5-formal。
assert (CLOUD / PHASE_TO_SAVE).is_dir()
checkpoint(PHASE_TO_SAVE)


## 15. 下载结果与 notebook，再释放 runtime
此格只在计时完全停止后运行。归档仅包括本实验结果、prepared 输入及公共来源/许可证、外部 checkpoint；不含依赖库、原始大包、Google 认证或 Drive。原始日志可能包含 VM 路径，下载备存不等于可以直接公开，应先用审计导出的便携文件并检查脱敏。

`files.download` 可能没有触发可取回的文件，不能看到调用成功就声称下载成功。确认本机文件及 SHA；另用 File → Download 保存 notebook。如果下载受阻，可用“查看笔记本 JSON”复制完整内容，UTF-8 保存，然后以 `scripts/recover_cloud_checkpoint.py` 恢复普通 checkpoint。参数 envelope 是单独的 raw-base64 格式，应按 `filename/bytes/sha256` 解码核验，恢复工具不会自动处理。下载不到之前不要释放 VM。

最终解释只比较**同机、同路线、同 dtype** 的 CPU/GPU 比值；CPU64→GPU32 同时改变精度，不能称 GPU 自身收益。Native 与 R/hybrid 的 RNG/流程不同，分别标注。CUDA 缺测不填值，Linux/T4 不代表 Windows/RTX3080。


In [ ]:
RUN_DOWNLOAD = False
assert RUN_DOWNLOAD, "手动启用下载；不要在计时期间压缩"
assert ACTIVE_PHASE is None and LIVE_PROCESS_GROUP is None
from google.colab import files

archive = pathlib.Path("/content") / ("amelia-records-" + RUN_ID + "-" + uuid.uuid4().hex[:6] + ".tar.gz")
assert not archive.exists()
with tarfile.open(archive, "w:gz") as bundle:
    for item in ("results/local/cloud", "data/prepared", "data/manifest.json", "LICENSE",
                 "THIRD_PARTY.md", "docs/cloud-cuda.md", "docs/validation/g5-prespecified/protocol.json"):
        path = REPO / item
        if path.exists():
            bundle.add(path, arcname=item)
    bundle.add(BACKUPS, arcname="checkpoint-envelopes")
digest = hashlib.sha256()
with archive.open("rb") as stream:
    for chunk in iter(lambda: stream.read(1024 * 1024), b""):
        digest.update(chunk)
manifest = {"artifact": archive.name, "bytes": archive.stat().st_size, "sha256": digest.hexdigest(),
            "revision": REVISION, "template_end_to_end_validated": False}
archive.with_suffix(archive.suffix + ".sha256.json").write_text(json.dumps(manifest, indent=2) + "\n")
print(manifest)
files.download(str(archive))
files.download(str(archive.with_suffix(archive.suffix + ".sha256.json")))


## 16. 只在本机离线恢复参数 envelope（说明代码，不是新增计算格）

普通 `amelia_checkpoint` 用现有 `recover_cloud_checkpoint.py`。另外，把 notebook 输出中十二个 `kind = native-parameter-raw-base64-v1` 的 JSON 各自复制成 `parameter-envelopes/*.json`，以下标准库代码恢复原 NPZ 字节；不执行 notebook、安装或拟合。先在本机创建这十二个 JSON 文件，输出目录必须是新的。它们是未脱敏的公共数据参数/样本字节，必须保持原 SHA；报告文本的脱敏哈希不是这个二进制哈希。

```python
import base64
import hashlib
import json
from pathlib import Path

source = Path("parameter-envelopes")
target = Path("recovered-native-parameters")
expected = {f"{dataset}-{method}.parameters.npz"
            for dataset in ("covertype", "household_power", "year_prediction_msd")
            for method in ("cpu64", "cpu32", "cuda64", "cuda32")}
records = [json.loads(path.read_text()) for path in sorted(source.glob("*.json"))]
assert len(records) == 12 and {row["filename"] for row in records} == expected
verified = {}
for row in records:
    assert row["kind"] == "native-parameter-raw-base64-v1"
    assert type(row["bytes"]) is int and 0 < row["bytes"] <= 64 * 1024 * 1024
    raw = base64.b64decode(row["data"], validate=True)
    assert len(raw) == row["bytes"]
    assert hashlib.sha256(raw).hexdigest() == row["sha256"]
    verified[row["filename"]] = raw  # Filename belongs to the fixed allowlist above.
target.mkdir()  # Refuse an existing directory; never overwrite earlier evidence.
for name, raw in verified.items():
    with (target / name).open("xb") as stream:
        stream.write(raw)
print("Recovered and hash-verified 12 original NPZ files")
```

恢复后把十二份原文件放进**已恢复 native suite 报告的同一目录**，保留原名字，再运行 `compare_native_parameters.py`；还需要相同 prepared 输入。参数哈希验证不证明整个矩阵一致，也不重新验证当时生成的随机数组。
